<a href="https://colab.research.google.com/github/steffenvogler/biomedical-courses/blob/main/croissant-deeplake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Croissant plus Deep Lake

In [7]:
!pip install deeplake mlcroissant scikit-image matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


##Restart Colab kernel

In [ ]:
# Make sure to restart Colab runtime after installing dependencies
import os
try:
    import google.colab
    os._exit(0)
except ImportError:
    pass

## Load libraries

In [8]:
from datetime import datetime
import json
import multiprocessing
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import os

import deeplake
#from google.colab import userdata
import matplotlib.pyplot as plt
import mlcroissant as mlc
from PIL import Image
from skimage import data, restoration, util

In [14]:
dataset = 'dcm_chest'
org_id = 'bay_224' # CHANGE THIS ACCORDING TO YOU ORG ON https://app.activeloop.ai/
path_to_deeplake_db = f'al://{org_id}/{dataset}'

path_to_croissant_file = f'/content/{dataset}.json'

For sake of example we just create our own Croissant file. But Croissant files are provided also in Huggingface, Kaggle, OpenML, and TFDS. If you want to share confidential, in-house data you can simply create your own Croissant (either programmatically as below or using the Croissant [Editor](https://huggingface.co/spaces/MLCommons/croissant-editor))

In [19]:
croissant_details = {
  "@context": {
    "@language": "en",
    "@vocab": "https://schema.org/",
    "arrayShape": "cr:arrayShape",
    "citeAs": "cr:citeAs",
    "column": "cr:column",
    "conformsTo": "dct:conformsTo",
    "cr": "http://mlcommons.org/croissant/",
    "data": {
      "@id": "cr:data",
      "@type": "@json"
    },
    "dataBiases": "cr:dataBiases",
    "dataCollection": "cr:dataCollection",
    "dataType": {
      "@id": "cr:dataType",
      "@type": "@vocab"
    },
    "dct": "http://purl.org/dc/terms/",
    "extract": "cr:extract",
    "field": "cr:field",
    "fileProperty": "cr:fileProperty",
    "fileObject": "cr:fileObject",
    "fileSet": "cr:fileSet",
    "format": "cr:format",
    "includes": "cr:includes",
    "isArray": "cr:isArray",
    "isLiveDataset": "cr:isLiveDataset",
    "jsonPath": "cr:jsonPath",
    "key": "cr:key",
    "md5": "cr:md5",
    "parentField": "cr:parentField",
    "path": "cr:path",
    "personalSensitiveInformation": "cr:personalSensitiveInformation",
    "recordSet": "cr:recordSet",
    "references": "cr:references",
    "regex": "cr:regex",
    "repeated": "cr:repeated",
    "replace": "cr:replace",
    "sc": "https://schema.org/",
    "separator": "cr:separator",
    "source": "cr:source",
    "subField": "cr:subField",
    "transform": "cr:transform"
  },
  "@type": "sc:Dataset",
  "distribution": [
    {
      "contentUrl": "https://huggingface.co/datasets/UniDataPro/chest-xrayresolve/main/data.zip?download=true",
      "contentSize": "54.4 MB",
      "md5": "4ba1656f0a0ea58094c8e55254ab808b",
      "encodingFormat": "application/zip",
      "@id": "data.zip",
      "@type": "cr:FileObject",
      "name": "data.zip",
      "description": "Chest X-rays, DICOM Data and Segmentation This dataset consists of 150 medical studies with chest X-ray (CXR) images primarily focused on the detection of lung diseases, including COVID-19 cases and pneumonias. The collection includes frontal chest radiographs and chest radiography scans in DICOM format. The dataset is ideal for medical research, disease detection, and classification tasks, particularly for developing computer-aided diagnosis and machine learning models. The dataset features annotations and segmentation results, with lung segmentations provided by radiologists and medical experts. These annotations are useful for training deep learning algorithms to improve classification performance in identifying common diseases and lung abnormalities."
    },
    {
      "includes": "*.txt",
      "containedIn": {
        "@id": "data.zip"
      },
      "encodingFormat": "text/txt",
      "@id": "image_labels",
      "@type": "cr:FileSet",
      "name": "image_labels"
    },
    {
      "includes": "*.**g",
      "containedIn": {
        "@id": "data.zip"
      },
      "encodingFormat": "image/dcm",
      "@id": "image-files",
      "@type": "cr:FileSet",
      "name": "image/dcm files",
      "description": "image/dcm files contained in data.zip"
    }
  ],
  "recordSet": [
    {
      "@type": "cr:RecordSet",
      "@id": "images",
      "name": "images",
      "key": {
        "@id": "img_id"
      },
      "field": [
        {
          "@type": "cr:Field",
          "@id": "images/image_filename",
          "name": "images/image_filename",
          "description": "The filename of the image. eg: COCO_train2014_000000000003.jpg",
          "dataType": "sc:Text",
          "source": {
            "fileSet": {
              "@id": "image-files"
            },
            "extract": {
              "fileProperty": "filename"
            }
          }
        },
        {
          "@type": "cr:Field",
          "@id": "images/image_content",
          "name": "images/image_content",
          "description": "The content of the image.",
          "dataType": "sc:ImageObject",
          "source": {
            "fileSet": {
              "@id": "image-files"
            },
            "extract": {
              "fileProperty": "content"
            }
          }
        },
          {
          "@type": "cr:Field",
          "@id": "images/label",
          "name": "images/label",
          "dataType": [
            "sc:Text"
          ],
          "source": {
            "fileSet": {
              "@id": "image-files"
            },
            "extract": {
              "fileProperty": "fullpath"
            },
            "transform": {
              "regex": "^.*/(.*)/.*..*$"
            }
          }
        }
      ]
    }
  ],
  "conformsTo": "http://mlcommons.org/croissant/1.1",
  "name": "Chest X-Ray",
  "description": "bliblublublbubjbjdfg",
  "alternateName": [
    "Genius-Society/HEp2",
    "HEp-2 Cell"
  ],
  "creator": {
    "@type": "Organization",
    "name": "Genius Society",
    "url": "https://huggingface.co/Genius-Society"
  },
  "keywords": [
    "image-classification",
    "English",
    "mit",
    "imagefolder",
    "Image",
    "Datasets",
    "Croissant",
    "arxiv:1504.02531",
    "🇺🇸 Region: US",
    "biology",
    "medical"
  ],
  "license": "https://choosealicense.com/licenses/mit/",
  "url": "https://huggingface.co/datasets/Genius-Society/HEp2"
}


with open(os.path.basename(path_to_croissant_file), "w") as f:
    json.dump(croissant_details, f, indent=4)

print(f"{path_to_croissant_file} created successfully!")

/content/dcm_chest.json created successfully!


In [25]:
dataset = mlc.Dataset(jsonld="dcm_chest.json")
metadata = dataset.metadata.to_json()

  -  [Metadata(Chest X-Ray)] Property "http://mlcommons.org/croissant/citeAs" is recommended, but does not exist.
  -  [Metadata(Chest X-Ray)] Property "https://schema.org/datePublished" is recommended, but does not exist.
  -  [Metadata(Chest X-Ray)] Property "https://schema.org/version" is recommended, but does not exist.


In [26]:
# Check out all available metadata items
print("## Available metadata ##")
print("\n")
for key in metadata:
  print(key)

print(2*"\n")
print(f"Name of the dataset: {metadata['name']}\n\nDescription: {metadata['description'][:15]} ...")

## Available metadata ##


@context
@type
name
description
conformsTo
creator
keywords
license
url
distribution
recordSet



Name of the dataset: Chest X-Ray

Description: bliblublublbubj ...


##Download data defined in croissant.json and save to Deeplake object

In [ ]:
records_loaded = dataset.records(record_set="images")

print("number of images in the dataset: {}".format(len(list(records_loaded))))

print("do sanity check...")
for i, record in enumerate(records_loaded):
  img = record['images/image_content']
  filename = record['images/image_filename'].decode("utf-8")
  img.save(filename, "PNG")
  print("index {} is file \"{}\" with label \"{}\" and image shape {}".format(i, record['images/image_filename'].decode("utf-8"), record['images/label'].decode("utf-8"), np.asarray(img).shape))
  if i > 2:
    break

TO-DO: Get an API token from https://app.activeloop.ai/ and add as a secret to the colab secret manager. Name it ACTIVELOOP_TOKEN

Below code is taking the Croissant 🥐 file meant for comprehensive data sharing and turns into a general-purpose Deep Lake object.

The best of two worlds 💪!

In [ ]:
start_time = datetime.now()

try:
    deeplake.delete(path_to_deeplake_db, token=userdata.get('ACTIVELOOP_TOKEN'))
except Exception as e:
    print(f"Could not delete dataset {path_to_deeplake_db}: {e}")
    pass

ds = deeplake.create(path_to_deeplake_db, token = userdata.get('ACTIVELOOP_TOKEN'))

num_cpu = multiprocessing.cpu_count()
print ("Processing with {} cpus".format(num_cpu))

for key in metadata:
  if key == 'recordSet': continue
  print(f"Adding Croissant metadata to deeplake DB: {key}")
  croissant_obj = metadata[key]
  if isinstance(croissant_obj, datetime):
    croissant_obj = croissant_obj.strftime("%Y-%m-%d %H:%M:%S.%f")
  ds.metadata[key] = croissant_obj

record_sets = ", ".join([f"`{rs.id}`" for rs in dataset.metadata.record_sets])

record_sets = [f"{rs.id}" for rs in dataset.metadata.record_sets]

for i in record_sets:
  ds.add_column("record_set", "text")
  ds.add_column("filename", "text")
  ds.add_column("label", "text")
  ds.add_column("image", deeplake.types.Image(sample_compression="png"))

  records_loaded = dataset.records(record_set=i)
  print("number of images in the dataset: {}".format(len(list(records_loaded))))
  for j,record in tqdm(enumerate(records_loaded), total=len(list(records_loaded))):
    arr = np.asarray(record['images/image_content'])
    new_arr = np.expand_dims(arr, axis=-1)

    if len(new_arr.shape) == 2: continue
    ds.append([{
        "record_set": i,
        "filename": record['images/image_filename'].decode("utf-8"),
        "label": record['images/label'].decode("utf-8"),
        "image": new_arr
    }])

ds.commit("initial commit after importing from Croissant")

stop_time = datetime.now()
execution_time = stop_time - start_time

print(f"Execution time: {execution_time}")
ds.summary()

Sanity check to load a file and display it

In [ ]:
i = 120

import PIL
from PIL import Image
import matplotlib.pyplot as plt

print(ds[i]["filename"])
print(ds[i]["label"])

array = ds[i]["image"]
img = Image.fromarray(np.squeeze(array))

plt.imshow(img)
plt.axis('off') # Hide axes
plt.show()

In [ ]:
print(ds.branches)

In [ ]:
# Create branch
branch = ds.branch("bg_subtraction")
# Open branch
branch_ds = branch.open()

In [ ]:
try:
  branch_ds.add_column("bg_subtracted", deeplake.types.Image(sample_compression="png"), )
  branch_ds.add_column("bg", deeplake.types.Image(sample_compression="png"))
except:
  print("Columns already exist")
  pass

column = branch_ds["image"]
for j,entry in tqdm(enumerate(column), total=len(list(column))):
  background = restoration.rolling_ball(entry, radius = 25)
  result = entry - background
  branch_ds[j]["bg_subtracted"] = result
  branch_ds[j]["bg"] = background
  if j > 100:
    break

branch_ds.metadata["python-function"] = "restoration.rolling_ball(entry, radius = 25)"
branch_ds.commit("rolling ball background subtraction")

branch_ds.summary()

In [ ]:
print(branch_ds.branches)
print(branch_ds.metadata.keys())

Sanity check whether background subtraction got actually saved in deep lake object.

In [ ]:
def plot_result(image, background, result):
    fig, ax = plt.subplots(nrows=1, ncols=3)

    ax[0].imshow(image)
    ax[0].set_title('Original image')
    ax[0].axis('off')

    ax[1].imshow(background)
    ax[1].set_title('Background')
    ax[1].axis('off')

    ax[2].imshow(result)
    ax[2].set_title('background subtracted result')
    ax[2].axis('off')

    fig.tight_layout()

i = 50

print("result for {}".format(branch_ds[i]["filename"]))

array = branch_ds[i]["image"]
original = Image.fromarray(np.squeeze(array))

array = branch_ds[i]["bg"]
bg = Image.fromarray(np.squeeze(array))

array = branch_ds[i]["bg_subtracted"]
bg_sub = Image.fromarray(np.squeeze(array))

plot_result(original, bg, bg_sub)
plt.show()

If desired, the newly created brach "bg_subtraction" can be merged into the main branch.

In [ ]:
print("before merging")
print(10*"#")
ds.summary()

ds.merge("bg_subtraction")
print("after merging")
print(10*"#")
ds.summary()

Never loose track of data lineage and if you wonder which data preprocessing would lead to an increased performance of you favorite classification model, then you are already on your way to do data-centric ML.